# Week 5 Practical: Multimodal Large Language Models

## Vision-Language Models: From CLIP to Building Your Own VLM

### Learning Objectives

By the end of this practical session, you will:

1. **Understand CLIP embeddings** (Exercise 1)
   - Explore how CLIP aligns images and text in a shared space
   - Perform zero-shot image classification
   - Analyze the embedding space with visualization

2. **Build a Vision-Language Model** (Exercise 2)
   - Understand how modern VLMs connect vision encoders to LLMs
   - Implement a visual projection layer
   - Use prompt tuning to improve modality alignment
   - Train your model on image captioning
   - Generate captions with your trained model

## Setup

The next cell imports the necessary modules – just run it, no need to look at
the details!

In [ ]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from io import BytesIO
from typing import List, Optional, Tuple
from master_mind.teaching.hf import load_hf_dataset, load_hf_model, load_hf_processor, load_hf_tokenizer


In [ ]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

In [ ]:
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

### Helper Functions

We'll use some utility functions to load and display images throughout this practical.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL."""
    response = requests.get(url, timeout=10)
    return Image.open(BytesIO(response.content)).convert("RGB")


def load_image(path_or_url: str) -> Image.Image:
    """Load an image from a local path or URL."""
    if path_or_url.startswith("http"):
        return load_image_from_url(path_or_url)
    return Image.open(path_or_url).convert("RGB")


def display_images(images: List[Image.Image], titles: List[str] = None, ncols: int = 4):
    """Display a grid of images with optional titles."""
    n = len(images)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    _, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

    # Ensure axes is always a flat list
    if nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i, (ax, img) in enumerate(zip(axes, images)):
        ax.imshow(img)
        ax.axis("off")
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=10)

    # Hide empty subplots
    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### Loading the Flickr8k Dataset

We'll use the Flickr8k dataset throughout this practical. It contains 8,000 images
with 5 captions each, making it ideal for learning multimodal alignment.

### Loading Flickr8k Dataset

In [ ]:
flickr_dataset = load_hf_dataset("jxie/flickr8k")
print(f"Dataset splits: {list(flickr_dataset.keys())}")
print(
    f"Train: {len(flickr_dataset['train'])} | Val: {len(flickr_dataset['validation'])} | Test: {len(flickr_dataset['test'])}"
)

In [ ]:
# Select sample images for visualization (use diverse samples from the dataset)
n_samples = 16
sample_indices = list(range(0, n_samples * 10, 10))  # Spread out samples
sample_data = flickr_dataset["train"].select(sample_indices)

sample_images = [item["image"] for item in sample_data]
sample_captions = [item["caption_0"] for item in sample_data]  # First caption of each

# Display some sample images with their captions
print(f"Loaded {len(sample_images)} sample images")
display_images(
    sample_images[:8], titles=[c[:40] + "..." for c in sample_captions[:8]], ncols=4
)

---

## Exercise 1: CLIP Embeddings and Zero-Shot Classification

[CLIP](https://openai.com/research/clip) (Contrastive Language-Image Pre-training)
learns to align images and text in a shared embedding space. This allows for
powerful zero-shot capabilities: we can classify images using arbitrary text labels
without any task-specific training.

### How CLIP Works

CLIP consists of two encoders:
- **Image encoder**: A Vision Transformer (ViT) that produces image embeddings
- **Text encoder**: A Transformer that produces text embeddings

During training, CLIP learns to maximize the similarity between matching image-text
pairs while minimizing similarity for non-matching pairs (contrastive learning with
InfoNCE loss).

### Load CLIP Model

We'll use the base CLIP model (`openai/clip-vit-base-patch32`) which is lightweight
enough to run on limited hardware (~600MB).

### Loading CLIP Model

In [ ]:
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = load_hf_model(clip_model_name, CLIPModel).to(device)
clip_processor = load_hf_processor(clip_model_name, CLIPProcessor, use_fast=True)

print(f"Model: {clip_model_name}")
print(
    f"Image encoder parameters: {sum(p.numel() for p in clip_model.vision_model.parameters()):,}"
)
print(
    f"Text encoder parameters: {sum(p.numel() for p in clip_model.text_model.parameters()):,}"
)
print(f"Embedding dimension: {clip_model.config.projection_dim}")

### Exercise 1.1: Computing CLIP Embeddings

Let's compute embeddings for our sample images and some text descriptions.
CLIP embeddings are normalized, so we can use cosine similarity (or just dot product)
to measure similarity.

In [ ]:
def get_image_embeddings(images: List[Image.Image], model, processor) -> torch.Tensor:
    """Compute CLIP embeddings for a list of images.

    Args:
        images: List of PIL images
        model: CLIP model
        processor: CLIP processor

    Returns:
        Normalized image embeddings of shape (n_images, embedding_dim)
    """
    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        embeddings = model.get_image_features(**inputs)
    # Normalize embeddings
    embeddings = F.normalize(embeddings, p=2, dim=-1)
    return embeddings


def get_text_embeddings(texts: List[str], model, processor) -> torch.Tensor:
    """Compute CLIP embeddings for a list of texts.

    Args:
        texts: List of text strings
        model: CLIP model
        processor: CLIP processor

    Returns:
        Normalized text embeddings of shape (n_texts, embedding_dim)
    """
    # Compute text embeddings using the CLIP model
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        embeddings = model.get_text_features(**inputs)
    embeddings = F.normalize(embeddings, p=2, dim=-1)
    return embeddings


# Test the functions
test_image = sample_images[0]
test_texts = ["a photo of a cat", "a photo of a dog"]

img_emb = get_image_embeddings([test_image], clip_model, clip_processor)
txt_emb = get_text_embeddings(test_texts, clip_model, clip_processor)

print(f"Image embedding shape: {img_emb.shape}")
print(f"Text embedding shape: {txt_emb.shape}")
print(f"Embedding dimension: {img_emb.shape[-1]}")

### Exercise 1.2: Zero-Shot Image Classification

With CLIP, we can classify images by computing the similarity between the image
embedding and embeddings of candidate text labels. The label with highest similarity
"wins".

In [ ]:
def zero_shot_classify(
    image: Image.Image,
    labels: List[str],
    model,
    processor,
    prompt_template: str = "a photo of a {}",
) -> Tuple[str, torch.Tensor]:
    """Classify an image using CLIP zero-shot classification.

    Args:
        image: PIL image to classify
        labels: List of candidate labels
        model: CLIP model
        processor: CLIP processor
        prompt_template: Template for creating text prompts (use {} for label)

    Returns:
        Tuple of (predicted label, probabilities for all labels)
    """
    prompts = [prompt_template.format(label) for label in labels]

    # Implement zero-shot classification

    # 1. Create text prompts from labels using the template
    # 2. Get image and text embeddings
    # 3. Compute similarities (dot product since embeddings are normalized)
    # 4. Convert to probabilities with softmax
    # 5. Return the label with highest probability
    assert False, 'Not implemented yet'


### Zero-shot classification results

In [ ]:
labels = ["person", "dog", "cat", "child", "water", "beach", "grass", "building"]

for i, img in enumerate(sample_images[:4]):
    pred_label, probs = zero_shot_classify(img, labels, clip_model, clip_processor)
    caption_short = sample_captions[i][:30] + "..."
    print(f"Image {i}: {caption_short:35} -> {pred_label:10} ({probs.max():.2%})")

### Exercise 1.3: Prompt Engineering for Zero-Shot Classification

The choice of prompt template significantly affects classification accuracy.
Let's experiment with different templates.

### Effect of prompt templates on classification

In [ ]:
# Different prompt templates to try
prompt_templates = [
    "a photo of a {}",
    "a {}",
    "an image of a {}",
    "a photograph showing a {}",
    "a picture of a {}",
]

# Use first sample image for template comparison
test_image = sample_images[0]
labels = ["person", "animal", "building", "nature"]

for template in prompt_templates:
    pred_label, probs = zero_shot_classify(
        test_image, labels, clip_model, clip_processor, prompt_template=template
    )
    print(f"Template: '{template:30}' -> Best: {pred_label:10} ({probs.max():.2%})")

### Exercise 1.4: Visualizing the CLIP Embedding Space

Let's visualize how images and their corresponding text descriptions are
positioned in the CLIP embedding space using PCA.

In [ ]:
# Compute embeddings for sample images and their captions
print("Computing embeddings for visualization...")
img_embeddings = get_image_embeddings(sample_images, clip_model, clip_processor)
txt_embeddings = get_text_embeddings(sample_captions, clip_model, clip_processor)

n_viz = len(sample_images)

# Compute cosine similarity matrix between images and texts
# This directly shows what CLIP learns: high similarity for matching pairs
similarity_matrix = (img_embeddings @ txt_embeddings.T).cpu().numpy()

# Plot similarity heatmap
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap="viridis", aspect="auto")
plt.colorbar(label="Cosine Similarity")
plt.xlabel("Caption Index")
plt.ylabel("Image Index")
plt.title("CLIP Image-Text Similarity Matrix\n(Diagonal = matching pairs)")

# Add diagonal line to highlight matching pairs
for i in range(n_viz):
    plt.plot(i, i, "rx", markersize=10, markeredgewidth=2)

plt.tight_layout()
plt.show()

# Print statistics
diagonal_sim = np.diag(similarity_matrix)
off_diagonal = similarity_matrix[~np.eye(n_viz, dtype=bool)]
print("\nSimilarity statistics:")
print(
    f"  Matching pairs (diagonal):     mean={diagonal_sim.mean():.3f}, std={diagonal_sim.std():.3f}"
)
print(
    f"  Non-matching pairs:            mean={off_diagonal.mean():.3f}, std={off_diagonal.std():.3f}"
)
print(
    f"  Separation (matching - other): {diagonal_sim.mean() - off_diagonal.mean():.3f}"
)

**Observation**: The similarity matrix shows CLIP's alignment: matching image-caption
pairs (on the diagonal, marked with red X) have higher similarity than non-matching
pairs. This is exactly what contrastive learning achieves - it brings matching pairs
together in the embedding space while pushing non-matching pairs apart.

---

## Exercise 2: Building a Vision-Language Model

Now that we understand how CLIP aligns images and text, let's build our own
Vision-Language Model (VLM) for image captioning. We'll connect CLIP's vision
encoder to a language model and train it to generate captions.

### Architecture Overview

Modern VLMs like LLaVA, Qwen-VL, and InstructBLIP follow a similar pattern:

```
┌─────────────────┐     ┌────────────────┐     ┌─────────────────┐
│  Vision Encoder │────▶│   Projection   │────▶│      LLM        │
│    (frozen)     │     │    + Prompts   │     │    (frozen)     │
└─────────────────┘     └────────────────┘     └─────────────────┘
     CLIP ViT           Trainable layers         Qwen/Llama/etc.
```

The key insight is that we only need to train a **projection layer** (and
optionally **soft prompts**) to bridge the vision and language modalities.
The heavy lifting is done by pre-trained models.

### 2.1 Load the Language Model

We'll use Qwen2.5-3B-Instruct as our language model. It's modern, capable,
and fits comfortably on GPUs with 8-16GB VRAM.

### Loading Language Model

In [ ]:
llm_model_name = "Qwen/Qwen2.5-3B-Instruct"


# Load tokenizer
llm_tokenizer = load_hf_tokenizer(llm_model_name, use_fast=True)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

# Load model in half precision to save memory
llm_model = load_hf_model(
    llm_model_name,
    AutoModelForCausalLM,
    dtype=torch.float16,
    device_map=device,
)
llm_model.eval()

print(f"Model: {llm_model_name}")
print(f"Parameters: {sum(p.numel() for p in llm_model.parameters()):,}")
print(f"Embedding dimension: {llm_model.config.hidden_size}")
print(f"Vocabulary size: {llm_model.config.vocab_size}")

### 2.2 Understanding Visual Features

Before building our projection, let's understand what CLIP's vision encoder outputs.
The ViT processes images as a sequence of patches, similar to how text is a sequence of tokens.

In [ ]:
# Get the hidden states from CLIP's vision encoder
test_img = sample_images[0]
inputs = clip_processor(images=test_img, return_tensors="pt").to(device)

with torch.no_grad():
    vision_outputs = clip_model.vision_model(**inputs, output_hidden_states=True)

# The output includes:
# - last_hidden_state: (batch, num_patches + 1, hidden_dim) - +1 for CLS token
# - pooler_output: (batch, hidden_dim) - the CLS token after projection

print("CLIP Vision Encoder Output:")
print(f"  Last hidden state shape: {vision_outputs.last_hidden_state.shape}")
print(f"  Pooler output shape: {vision_outputs.pooler_output.shape}")
print(f"  Number of patches: {vision_outputs.last_hidden_state.shape[1] - 1}")
print(f"  CLIP hidden dimension: {vision_outputs.last_hidden_state.shape[-1]}")
print(f"  LLM hidden dimension: {llm_model.config.hidden_size}")

### 2.3 Implement the Vision-Language Model

We'll implement a VLM that:
1. Extracts visual features using CLIP's vision encoder
2. Projects them to the LLM's embedding space
3. Adds learnable "soft prompt" tokens to help with modality alignment
4. Concatenates visual tokens with text tokens for caption generation

In [ ]:
class VisionLanguageModel(nn.Module):
    """A simple Vision-Language Model for image captioning.

    Architecture:
        Image -> CLIP ViT -> Projection MLP -> [Visual Tokens]
                                                     |
        [Soft Prompts] + [Visual Tokens] + [Text] -> LLM -> Caption
    """

    def __init__(
        self,
        clip_model: CLIPModel,
        llm_model: AutoModelForCausalLM,
        llm_tokenizer: AutoTokenizer,
        num_soft_prompts: int = 8,
        projection_hidden_dim: int = None,
    ):
        super().__init__()

        self.clip_vision = clip_model.vision_model
        self.llm = llm_model
        self.tokenizer = llm_tokenizer

        # Implement the projection MLP

        # Create a 2-layer MLP that maps from clip_hidden_dim to llm_hidden_dim
        # Use GELU activation between layers
        assert False, 'Not implemented yet'


        # Soft prompts: learnable tokens that help bridge modalities
        # Initialize learnable soft prompt embeddings

        # Create a learnable parameter of shape (num_soft_prompts, llm_hidden_dim)
        # Initialize with small random values (std=0.02)
        # Don't forget to freeze CLIP and LLM for now
        assert False, 'Not implemented yet'


    def get_visual_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Extract and project visual features from images.

        Args:
            pixel_values: Preprocessed images (batch, channels, height, width)

        Returns:
            Projected visual features (batch, num_patches, llm_hidden_dim)
        """
        # Get CLIP vision features
        with torch.no_grad():
            vision_outputs = self.clip_vision(pixel_values)
            # Use all patch tokens (exclude CLS token at position 0)
            visual_features = vision_outputs.last_hidden_state[:, 1:, :]

        # Project to LLM dimension
        visual_features = self.projection(
            visual_features.to(self.projection[0].weight.dtype)
        )
        return visual_features

    def get_soft_prompts(self, batch_size: int) -> torch.Tensor:
        """Get soft prompts expanded for batch.

        Args:
            batch_size: Number of samples in batch

        Returns:
            Soft prompts (batch, num_soft_prompts, llm_hidden_dim)
        """
        return self.soft_prompts.unsqueeze(0).expand(batch_size, -1, -1)

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> dict:
        """Forward pass for training.

        Args:
            pixel_values: Preprocessed images
            input_ids: Tokenized caption (without image tokens)
            attention_mask: Attention mask for caption
            labels: Labels for language modeling (shifted input_ids)

        Returns:
            Dictionary with loss and logits
        """
        batch_size = pixel_values.shape[0]

        # Get text embeddings from LLM (determines target dtype)
        text_embeds = self.llm.get_input_embeddings()(input_ids)
        target_dtype = text_embeds.dtype

        # Get visual features and soft prompts (cast to match LLM dtype)
        visual_features = self.get_visual_features(pixel_values).to(target_dtype)
        soft_prompts = self.get_soft_prompts(batch_size).to(target_dtype)

        # Concatenate: [soft_prompts, visual_features, text_embeds]
        # Concatenate the embeddings in the correct order

        # The sequence should be: soft prompts, then visual features, then text
        assert False, 'Not implemented yet'


        return {
            "loss": outputs.loss,
            "logits": outputs.logits,
        }

    @torch.no_grad()
    def generate(
        self,
        pixel_values: torch.Tensor,
        max_new_tokens: int = 50,
        temperature: float = 0.7,
        top_p: float = 0.9,
        do_sample: bool = True,
    ) -> List[str]:
        """Generate captions for images.

        Args:
            pixel_values: Preprocessed images
            max_new_tokens: Maximum tokens to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
            do_sample: Whether to use sampling (vs greedy)

        Returns:
            List of generated captions
        """
        batch_size = pixel_values.shape[0]

        # Add a prompt to guide generation (also determines target dtype)
        prompt = "This image shows"
        prompt_ids = self.tokenizer(prompt, return_tensors="pt").input_ids.to(
            pixel_values.device
        )
        prompt_embeds = self.llm.get_input_embeddings()(prompt_ids)
        prompt_embeds = prompt_embeds.expand(batch_size, -1, -1)
        target_dtype = prompt_embeds.dtype

        # Get visual features and soft prompts (cast to match LLM dtype)
        visual_features = self.get_visual_features(pixel_values).to(target_dtype)
        soft_prompts = self.get_soft_prompts(batch_size).to(target_dtype)

        # Concatenate: [soft_prompts, visual_features, prompt_embeds]
        inputs_embeds = torch.cat([soft_prompts, visual_features, prompt_embeds], dim=1)

        # Create attention mask
        attention_mask = torch.ones(
            batch_size, inputs_embeds.shape[1], device=pixel_values.device
        )

        # Generate
        outputs = self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else 1.0,
            top_p=top_p if do_sample else 1.0,
            do_sample=do_sample,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )

        # Decode (outputs include the prompt, so we take everything)
        captions = []
        for output in outputs:
            caption = self.tokenizer.decode(output, skip_special_tokens=True)
            # Prepend our prompt since it was in embeddings
            caption = prompt + " " + caption
            captions.append(caption)

        return captions


# Sanity checks for VisionLanguageModel implementation
def test_vlm_implementation():
    """Verify student implementation of VisionLanguageModel."""
    # Create a minimal test instance
    _vlm = VisionLanguageModel(
        clip_model=clip_model,
        llm_model=llm_model,
        llm_tokenizer=llm_tokenizer,
        num_soft_prompts=4,
    )

    # Test 1: Projection layer outputs correct dimension
    _test_clip_features = torch.randn(2, 10, _vlm.clip_hidden_dim)
    _projected = _vlm.projection(_test_clip_features)
    assert (
        _projected.shape[-1] == _vlm.llm_hidden_dim
    ), f"Projection output dim {_projected.shape[-1]} != LLM hidden dim {_vlm.llm_hidden_dim}"

    # Test 2: Soft prompts have correct shape
    assert hasattr(_vlm, "soft_prompts"), "Missing soft_prompts parameter"
    assert (
        _vlm.soft_prompts.shape == (4, _vlm.llm_hidden_dim)
    ), f"Soft prompts shape {_vlm.soft_prompts.shape} != expected (4, {_vlm.llm_hidden_dim})"

    # Test 3: CLIP and LLM should be frozen
    for param in _vlm.clip_vision.parameters():
        assert not param.requires_grad, "CLIP vision encoder should be frozen"
    for param in _vlm.llm.parameters():
        assert not param.requires_grad, "LLM should be frozen"

    # Test 4: Forward pass produces expected output structure
    _test_image = clip_processor(
        images=sample_images[0], return_tensors="pt"
    ).pixel_values
    _test_ids = llm_tokenizer(
        "test caption", return_tensors="pt", padding="max_length", max_length=10
    )
    _vlm = _vlm.to(device)
    _outputs = _vlm(
        pixel_values=_test_image.to(device),
        input_ids=_test_ids.input_ids.to(device),
        attention_mask=_test_ids.attention_mask.to(device),
        labels=_test_ids.input_ids.to(device),
    )
    assert "loss" in _outputs, "Forward pass must return 'loss'"
    assert "logits" in _outputs, "Forward pass must return 'logits'"
    assert _outputs["loss"] is not None, "Loss should not be None when labels provided"

    print("✓ VisionLanguageModel implementation verified!")


test_vlm_implementation()

### 2.4 Create the Model

Let's instantiate our VLM and check the number of trainable parameters.

### Creating Vision-Language Model

In [ ]:
vlm = VisionLanguageModel(
    clip_model=clip_model,
    llm_model=llm_model,
    llm_tokenizer=llm_tokenizer,
    num_soft_prompts=8,
)
vlm = vlm.to(device)

# Count parameters
total_params = sum(p.numel() for p in vlm.parameters())
trainable_params = sum(p.numel() for p in vlm.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable ratio: {trainable_params / total_params:.2%}")
print("\nTrainable components:")
print(f"  - Projection MLP: {sum(p.numel() for p in vlm.projection.parameters()):,}")
print(f"  - Soft prompts: {vlm.soft_prompts.numel():,}")

### 2.5 Prepare Training Data

We'll use subsets of the Flickr8k dataset (already loaded earlier) for training.

### Preparing Training Data

In [ ]:
# For faster training, we'll use a subset
train_size = 800
val_size = 200


# Use subsets of the official splits (flickr_dataset was loaded earlier)
train_dataset = flickr_dataset["train"].shuffle(seed=42).select(range(train_size))
val_dataset = flickr_dataset["validation"].shuffle(seed=42).select(range(val_size))

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

### 2.6 Create Data Loaders

We need to preprocess images with CLIP and tokenize captions for the LLM.

In [ ]:
class CaptionDataset(torch.utils.data.Dataset):
    """Dataset for image captioning."""

    def __init__(self, hf_dataset, clip_processor, llm_tokenizer, max_length=64):
        self.dataset = hf_dataset
        self.clip_processor = clip_processor
        self.llm_tokenizer = llm_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # Get image
        image = sample["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Process image for CLIP
        pixel_values = self.clip_processor(
            images=image, return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Get caption (use first caption - dataset has caption_0 through caption_4)
        caption = sample["caption_0"]

        # Tokenize caption
        # We add EOS token to mark end of caption
        caption_with_eos = caption + self.llm_tokenizer.eos_token
        encoding = self.llm_tokenizer(
            caption_with_eos,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        input_ids = encoding.input_ids.squeeze(0)
        attention_mask = encoding.attention_mask.squeeze(0)

        # Labels are the same as input_ids (causal LM)
        # We'll mask padding tokens with -100
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


# Create datasets and dataloaders
train_caption_dataset = CaptionDataset(
    train_dataset, clip_processor, llm_tokenizer, max_length=64
)
val_caption_dataset = CaptionDataset(
    val_dataset, clip_processor, llm_tokenizer, max_length=64
)

batch_size = 4  # Adjust based on your GPU memory


train_loader = DataLoader(
    train_caption_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    val_caption_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

### 2.7 Training Loop

Now let's train our model! We'll only update the projection layer and soft prompts.

In [ ]:
def train_epoch(model, dataloader, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0

    pbar = tqdm(dataloader, desc="Training")
    for batch in pbar:
        # Move batch to device
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        # Implement the forward pass and compute the loss

        # Call model() with pixel_values, input_ids, attention_mask, labels
        # Extract the loss from the outputs dictionary
        assert False, 'Not implemented yet'


        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        # Update progress bar with running loss
        pbar.set_postfix({"loss": f"{total_loss / num_batches:.4f}"})

    return total_loss / num_batches


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    num_batches = 0

    for batch in tqdm(dataloader, desc="Evaluating"):
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        total_loss += outputs["loss"].item()
        num_batches += 1

    return total_loss / num_batches

### 2.8 Train the Model

Let's train for a few epochs. With our small projection layer and soft prompts,
training should be fast even on modest hardware.

### Training Vision-Language Model

In [ ]:
# Training configuration
num_epochs = 3
learning_rate = 1e-4


# Only optimize trainable parameters
optimizer = torch.optim.AdamW(
    [p for p in vlm.parameters() if p.requires_grad],
    lr=learning_rate,
)

# Training loop
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss = train_epoch(vlm, train_loader, optimizer, device)
    val_loss = evaluate(vlm, val_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")

# Plot training curves
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses, "b-o", label="Train Loss")
plt.plot(range(1, num_epochs + 1), val_losses, "r-o", label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 2.9 Generate Captions

Now let's use our trained model to generate captions for images!

### Generating Captions

In [ ]:
def generate_caption_for_image(model, image, clip_processor, device):
    """Generate a caption for a single image."""
    # Preprocess image
    pixel_values = clip_processor(images=image, return_tensors="pt").pixel_values
    pixel_values = pixel_values.to(device)

    # Generate
    captions = model.generate(
        pixel_values,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
    )
    return captions[0]

### Generated captions for sample images

In [ ]:
num_samples_to_show = 6


for i, img in enumerate(sample_images[:num_samples_to_show]):
    caption = generate_caption_for_image(vlm, img, clip_processor, device)
    print(f"\n[Image {i}] Ground truth: {sample_captions[i][:50]}...")
    print(f"  Generated: {caption}")

# Display images with their generated captions
num_display = 4


display_images(
    sample_images[:num_display],
    titles=[
        generate_caption_for_image(vlm, img, clip_processor, device)[:50] + "..."
        for img in sample_images[:num_display]
    ],
    ncols=2,
)

### 2.10 Compare with Greedy vs Sampling

Let's compare different generation strategies.

### Comparing generation strategies

In [ ]:
test_img = sample_images[5]  # Use a different sample image
display_images([test_img], titles=["Test image"])

# Greedy decoding
pixel_values = clip_processor(images=test_img, return_tensors="pt").pixel_values.to(
    device
)

greedy_caption = vlm.generate(
    pixel_values,
    max_new_tokens=50,
    do_sample=False,
)[0]
print(f"\nGreedy: {greedy_caption}")

# Sampling with different temperatures
temperatures = [0.5, 0.7, 1.0]


for temp in temperatures:
    caption = vlm.generate(
        pixel_values,
        max_new_tokens=50,
        temperature=temp,
        do_sample=True,
    )[0]
    print(f"Sampling (T={temp}): {caption}")

### 2.11 Understanding Soft Prompts (Prompt Tuning)

Let's visualize what our soft prompts have learned by looking at their similarity
to vocabulary tokens.

### Analyzing Soft Prompts

In [ ]:
# Get the LLM's embedding matrix
embedding_matrix = (
    llm_model.get_input_embeddings().weight.data
)  # (vocab_size, hidden_dim)

# Compute similarity between soft prompts and vocabulary embeddings
soft_prompt_embeds = vlm.soft_prompts.data  # (num_prompts, hidden_dim)

# Normalize for cosine similarity
embedding_matrix_norm = F.normalize(embedding_matrix.float(), dim=-1)
soft_prompt_norm = F.normalize(soft_prompt_embeds.float(), dim=-1)

# Compute similarities
similarities = soft_prompt_norm @ embedding_matrix_norm.T  # (num_prompts, vocab_size)

# Find the most similar tokens for each soft prompt
print("Most similar vocabulary tokens to each soft prompt:")
print("-" * 50)

for i in range(vlm.num_soft_prompts):
    top_k = 5
    top_similarities, top_indices = similarities[i].topk(top_k)

    tokens = [llm_tokenizer.decode([idx.item()]) for idx in top_indices]
    sims = [f"{s.item():.3f}" for s in top_similarities]

    print(f"Soft prompt {i}: {list(zip(tokens, sims))}")

**Interpretation**: The soft prompts learn to represent concepts that help bridge
the visual and language modalities. They often correspond to words related to
visual description, image content, or scene understanding.

### Exercise 2.12: Experiment with Different Configurations

Try modifying the model configuration and observe the effects:

1. **Number of soft prompts**: Try `num_soft_prompts=4` vs `num_soft_prompts=16`
2. **Projection architecture**: Try a deeper MLP or a single linear layer
3. **Visual tokens**: Currently we use patch tokens. Try using only the CLS token
4. **Learning rate**: Experiment with different learning rates